# 第25章　データ拡張を実装する ― 2Dと3Dのレシピ

**『医療診断支援AIを自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 医療画像で最も効く一手 ― 弾性変形

In [ ]:
import albumentations as A

train_tf = A.Compose([
    # ---- 幾何変換：画像とマスクの両方に、同じものがかかる ----
    A.HorizontalFlip(p=0.5),                       # ※左右が診断に効く部位では使わない
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,   # 新しい版では A.Affine(...) が推奨
                       rotate_limit=15, p=0.7),
    A.ElasticTransform(alpha=30, sigma=6, p=0.3),  # 弾性変形（生体の形のばらつきを模す）
    # ---- 強度変換：画像だけにかかる（マスクは変わらない）----
    A.RandomBrightnessContrast(0.15, 0.15, p=0.5),
    A.RandomGamma(gamma_limit=(85, 115), p=0.3),   # ガンマ補正（装置ごとの階調差を模す）
    # ノイズ（低線量CTのざらつき）。albumentations 2.x の書式。std_range は最大値に対する比で、
    # (0.01, 0.02) は 1.x の var_limit=(5, 25)（標準偏差2.2〜5.0）にほぼ相当する。
    # 2.x に var_limit を渡すと警告が出るだけで黙って無視され、既定の std_range=(0.2, 0.44)
    # ― 桁違いに強く、画像が0と1に飽和する強度 ― がかかるので注意。
    A.GaussNoise(std_range=(0.01, 0.02), p=0.3),
])

out = train_tf(image=img, mask=mask)               # mask= に渡すだけで幾何変換が連動する
aug_img, aug_mask = out["image"], out["mask"]

## 25.2　2次元 ― torchvisionで

In [ ]:
from torchvision.transforms import v2
import torch

train_tf = v2.Compose([
    v2.ToImage(),                                # PILやndarrayをテンソルへ（val_tfと同じ）
    v2.RandomResizedCrop(224, scale=(0.8, 1.0)),
    v2.RandomRotation(15),                       # ±15度
    v2.ColorJitter(brightness=0.1, contrast=0.1),# 控えめに
    v2.RandomHorizontalFlip(p=0.5),              # ※左右のある臓器では使わない
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],     # 学習も検証と同じ正規化（欠かすと精度が落ちる）
                 std=[0.229, 0.224, 0.225]),
])
val_tf = v2.Compose([                            # 検証は拡張なし
    v2.ToImage(),                                # PILやndarrayをテンソルへ
    v2.Resize((224, 224)),                       # ※Resize(224)だと短辺だけ224になり、形が揃わない
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],     # 事前学習と同じ正規化（忘れると精度が落ちる）
                 std=[0.229, 0.224, 0.225]),
])

## 25.3　3次元 ― MONAIで

In [ ]:
from monai import transforms as T

train_tf3d = T.Compose([
    # RAS基準では 空間軸0=左右, 軸1=前後, 軸2=頭尾。頭尾（軸2）の反転は「頭と足が入れ替わった」
    # 現実にありえない画像を作るので使わない。左右（軸0）も、腹部のように臓器の左右が診断に
    # 効く部位では使わない。四肢のように左右どちらも実在する部位に限って許す。
    T.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),   # 左右のみ（下表で可否を判断）
    # 90度回転は横断面の向きを変える。RAS正規化後の体幹CT/MRIでは実際には得られない
    # 向きになるので、向きの自由度がある対象（四肢・単純X線・病理パッチ）に限って使う。
    T.RandRotate90d(keys=["image", "label"], prob=0.3, spatial_axes=(0, 1)),
    T.RandScaleIntensityd(keys="image", factors=0.1, prob=0.3),       # 明るさ
    T.RandGaussianNoised(keys="image", prob=0.2, std=0.02),           # ノイズ
    T.RandAffined(keys=["image", "label"], prob=0.3,                  # アフィン変形（回転・スケール）
                  # rotate_range の単位はラジアン（0.1 rad は約5.7度）。前節の rotate_limit=15 /
                  # RandomRotation(15) は「度」なので、同じ感覚で 15 と書くと約859度になり、
                  # エラーも出ないまま拡張が壊れる。またスカラーを渡すと空間軸0にしか回転が
                  # かからない ― 3軸すべてに効かせたいなら rotate_range=(0.1, 0.1, 0.1) と書く。
                  rotate_range=0.1, scale_range=0.1,
                  mode=("bilinear", "nearest")),                       # ラベルは最近傍
    # 局所的な非剛体ワープ（弾性変形）を試すなら T.Rand3DElasticd を使う
])

## 25.4　MixUp ― 2枚を混ぜる

In [ ]:
import numpy as np, torch

def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    mixed_x = lam * x + (1 - lam) * x[idx]       # 画像を混ぜる
    return mixed_x, y, y[idx], lam               # 2つのラベルと混合比を返す

# 使い方：損失も、2つのラベルで同じ比率に按分する
mixed_x, ya, yb, lam = mixup(x, y)
out = model(mixed_x)
loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)

## 数字で追う ― 各変換が画素をどう書き換えるか

In [ ]:
import numpy as np
patch = np.array([[0.0, 0.1, 0.2, 0.3],
                  [0.4, 0.5, 0.6, 0.7],
                  [0.8, 0.9, 1.0, 0.9],
                  [0.7, 0.5, 0.3, 0.1]])
mask  = np.array([[1, 1, 0, 0],
                  [1, 1, 0, 0],
                  [1, 1, 0, 0],
                  [1, 1, 0, 0]])

flip   = patch[:, ::-1]            # 左右反転：列の順を逆に並べ替える
rot90  = np.rot90(patch)          # 90度回転：軸を入れ替える
bright = np.clip(patch * 1.3, 0, 1)  # 明るさ+30%：値を一律に持ち上げ、1で頭打ち
rng    = np.random.default_rng(0)
noisy  = np.clip(patch + rng.normal(0, 0.05, patch.shape), 0, 1)  # ガウスノイズ